# Exploring the labeled issue reports

Before training any model we look at the data: how much of it there is, whether it is clean, and how the categories are distributed.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

data = pd.read_csv("../data/issues_labeled.csv")
data.head()

## Shape, types and missing values

In [ ]:
print(data.shape)
data.info()

In [ ]:
print("Missing values per column:")
print(data.isna().sum())
print()
print("Exact duplicate reports:", data.duplicated(subset=["title", "description"]).sum())

## How balanced are the categories?

A model that always answers the largest category gets this share right without learning anything. That share is the score to beat.

In [ ]:
counts = data["category"].value_counts()
shares = data["category"].value_counts(normalize=True).round(3)
pd.DataFrame({"reports": counts, "share": shares})

In [ ]:
figure, axes = plt.subplots(figsize=(7, 3.5))
counts.plot.bar(ax=axes, color="#1f6fb2")
axes.set_title("Reports per category")
axes.set_ylabel("Reports")
axes.tick_params(axis="x", rotation=20)
figure.tight_layout()
plt.show()

## How long are the reports?

In [ ]:
data["words"] = (data["title"] + " " + data["description"]).str.split().str.len()
data["words"].describe().round(1)

In [ ]:
data.groupby("category")["words"].agg(["mean", "median", "std", "min", "max"]).round(1)

In [ ]:
figure, axes = plt.subplots(figsize=(7, 3.5))
axes.hist(data["words"], bins=25, color="#1f6fb2")
axes.set_title("Length of a report in words")
axes.set_xlabel("Words")
axes.set_ylabel("Reports")
figure.tight_layout()
plt.show()

## Are single words enough to separate the categories?

If one word gave the answer away, the problem would be trivial and a model would learn nothing useful.

In [ ]:
text = (data["title"] + " " + data["description"]).str.lower()
for word in ["slow", "security", "docs", "crash", "add"]:
    share = text.str.contains(word, regex=False).groupby(data["category"]).mean().round(2)
    print(f"{word!r:12}", share.to_dict())